# 11b - Follow-up: late fusion with every text encoder

**This is a follow-up, added after seeing the results of notebook 11; it was not part of
the pre-set plan, and is reported as such.**

**Why:** notebook 11 compared the six text encoders with concatenation fusion only. The
language models then failed the requirement (worse than ResNet-50 alone when no text is
given) and showed a healthy-text bias. Late fusion avoids both by construction: with no
text it *is* ResNet-50 alone, and text can only re-rank the disease classes, never move
the total probability of "healthy". So this run asks the remaining question: when a farmer
does type a description, does a good language model help?

**Rule, fixed before running** (reference numbers copied from notebook 11's results):
1. *Requirement:* field-validation macro-F1 with no text at least ResNet-50 alone minus 0.5
   points (met by construction; still computed).
2. *Healthy-bias cap:* each bias rate (healthy photo + disease text pushed to a disease;
   healthy photo + same-crop text pushed to a disease; diseased photo without text called
   healthy) at most the current winner's rate (TF-IDF + concatenation) **plus 1 point**.
3. *Replacement:* the best eligible setup replaces the current winner only if its
   field-validation log-loss with text is lower **beyond the seed spread** (mean plus one
   standard deviation below the current winner's mean minus one). Otherwise TF-IDF +
   concatenation stays.

Late fusion trains only a text classifier on the training sentences, so the missing-text
rate does not apply. Uses the image features cached by notebook 11; about 10 minutes on a T4.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings and the reference numbers from notebook 11

In [ ]:
import importlib
import json

import numpy as np
import pandas as pd
import torch

from src import fusion, metrics, text_encoders, training
for module in (fusion, metrics, text_encoders, training):
    importlib.reload(module)

ENCODERS = list(text_encoders.ENCODERS)
SEEDS = [42, 43, 44]
H1_MARGIN = 0.005
BIAS_CAP_MARGIN = 0.01
FIELD_SOURCES = ["fieldplant", "tanzania"]

# Current winner in notebook 11 (TF-IDF + concatenation, 20% missing text), means over its 3 seeds
# on the field validation photos (ml/results/fusion/stage1_encoders.csv).
CURRENT = {"name": "tfidf + concat", "nll_mean": 0.028232, "nll_sd": 0.002601,
           "healthy_with_disease_text": 0.003268, "healthy_with_same_crop_text": 0.003268,
           "diseased_without_text_called_healthy": 0.004372}
BIAS_KEYS = ["healthy_with_disease_text", "healthy_with_same_crop_text", "diseased_without_text_called_healthy"]

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"
FEATURES_DIR = config.MODELS_DIR / "fusion_features"
OUT_DIR = config.MODELS_DIR / "fusion_late_followup"
OUT_DIR.mkdir(parents=True, exist_ok=True)
training.seed_everything(config.RANDOM_SEED)

## Load the cached image features and the sentences

In [ ]:
CLASSES = list(config.PLANTVILLAGE_CLASSES)
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in ["train", "val", "test", "ood_test"]}

def load(name):
    path = FEATURES_DIR / f"{name}.npz"
    assert path.exists(), f"{path} missing: run notebook 11 (QUICK_TEST = False) first"
    saved = np.load(path)
    return fusion.Features(saved["image"], saved["logits"], saved["labels"])

feats = {name: load(name) for name in splits}
for name in splits:
    assert len(feats[name].labels) == len(splits[name]), f"{name}: features do not match the split"

def field_part(name):
    mask = splits[name]["source"].isin(FIELD_SOURCES).to_numpy()
    f = feats[name]
    return fusion.Features(f.image[mask], f.logits[mask], f.labels[mask])

val_field = field_part("val")
RESNET = metrics.summarise(val_field.logits, val_field.labels, len(CLASSES))
print(f"ResNet-50 alone, field validation: macro-F1 {RESNET['macro_f1']:.4f}, log-loss {RESNET['nll']:.4f}")

descriptions = pd.read_csv(ML_DIR / "data" / "symptoms" / "descriptions.csv", encoding="utf-8-sig")
descriptions = descriptions[descriptions["keep"] == "y"].reset_index(drop=True)
texts = descriptions["text"].tolist()
train_texts = descriptions.loc[descriptions["split"] == "train", "text"].tolist()
banks = {}
for name in ENCODERS:
    vectors = text_encoders.encode(name, texts, train_texts, DEVICE)
    banks[name] = {split: fusion.TextBank.build(descriptions, vectors, CLASSES, split) for split in ["train", "val", "test"]}
    torch.cuda.empty_cache()
print("Encoded sentences for", ENCODERS)

## Late fusion with every encoder, 3 seeds each

In [ ]:
def no_text(features, bank):
    return np.zeros((len(features.labels), bank.dim), np.float32), np.zeros(len(features.labels), np.float32)

rows = []
for encoder in ENCODERS:
    bank = banks[encoder]
    for seed in SEEDS:
        head = fusion.train_head("late", feats["train"], val_field, bank["train"], bank["val"], CLASSES, 0.0, seed, DEVICE)
        without = metrics.summarise(fusion.predict("late", head, val_field, *no_text(val_field, bank["val"]), CLASSES, DEVICE),
                                    val_field.labels, len(CLASSES))
        with_text = metrics.summarise(fusion.predict("late", head, val_field, *bank["val"].fixed(val_field.labels), CLASSES, DEVICE),
                                      val_field.labels, len(CLASSES))
        rows.append({"encoder": encoder, "seed": seed, "f1_no_text": without["macro_f1"], "f1_with_text": with_text["macro_f1"],
                     "nll_with_text": with_text["nll"], "ece_with_text": with_text["ece"],
                     **fusion.bias_tests("late", head, val_field, bank["val"], CLASSES, DEVICE)})
followup = pd.DataFrame(rows)
followup.to_csv(OUT_DIR / "late_all_encoders.csv", index=False)

## Apply the rule

In [ ]:
summary = followup.groupby("encoder").agg(
    f1_no_text=("f1_no_text", "mean"), nll=("nll_with_text", "mean"), nll_sd=("nll_with_text", "std"),
    **{key: (key, "mean") for key in BIAS_KEYS}).fillna({"nll_sd": 0.0})
summary["meets_H1"] = summary["f1_no_text"] >= RESNET["macro_f1"] - H1_MARGIN
summary["within_bias_cap"] = np.all([summary[k] <= CURRENT[k] + BIAS_CAP_MARGIN for k in BIAS_KEYS], axis=0)
summary["beats_current_beyond_spread"] = summary["nll"] + summary["nll_sd"] < CURRENT["nll_mean"] - CURRENT["nll_sd"]
summary["eligible"] = summary["meets_H1"] & summary["within_bias_cap"] & summary["beats_current_beyond_spread"]
display(summary.round(4))

eligible = summary[summary["eligible"]]
if eligible.empty:
    DECISION = {"replace_current": False, "winner": CURRENT["name"],
                "reason": "no late-fusion setup met the bias cap and beat the current winner beyond the seed spread"}
else:
    best = eligible["nll"].idxmin()
    DECISION = {"replace_current": True, "winner": f"{best} + late",
                "reason": f"lowest log-loss with text among eligible setups ({eligible.loc[best, 'nll']:.4f} vs {CURRENT['nll_mean']:.4f})"}
print(json.dumps(DECISION, indent=1))

## If a late-fusion setup wins: test and PlantDoc

Reported only for a winner. Image only (no text) and with a held-out test sentence (the
controlled upper bound); PlantDoc is reported, never used to choose.

In [ ]:
final_rows, test_bias = [], None
if DECISION["replace_current"]:
    encoder = DECISION["winner"].split(" + ")[0]
    bank = banks[encoder]
    head = fusion.train_head("late", feats["train"], val_field, bank["train"], bank["val"], CLASSES, 0.0, SEEDS[0], DEVICE)
    for label, part in [("test", feats["test"]), ("test: field", field_part("test")), ("ood_test", feats["ood_test"])]:
        final_rows.append({"set": label, "condition": "ResNet-50 alone", **metrics.summarise(part.logits, part.labels, len(CLASSES))})
        final_rows.append({"set": label, "condition": "late fusion, with test text",
                           **metrics.summarise(fusion.predict("late", head, part, *bank["test"].fixed(part.labels), CLASSES, DEVICE),
                                               part.labels, len(CLASSES))})
    test_bias = fusion.bias_tests("late", head, field_part("test"), bank["test"], CLASSES, DEVICE)
    torch.save(head.state_dict(), OUT_DIR / "late_head.pt")
    final = pd.DataFrame(final_rows).set_index(["set", "condition"])
    display(final.round(4))
    final.to_csv(OUT_DIR / "final_metrics.csv")
    display(pd.Series(test_bias).round(3))
else:
    print("Current winner kept; nothing more to evaluate.")
training.save_json({**DECISION, "current": CURRENT, "resnet50_val_field": RESNET, "test_bias": test_bias,
                    "post_hoc_followup": True}, OUT_DIR / "decision.json")

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
import shutil

REPO_RESULTS = ML_DIR / "results" / "fusion_late_followup"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "11b_late_fusion_followup.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/fusion-late-followup",
    message="Follow-up: late fusion with every text encoder",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))